<a href="https://colab.research.google.com/github/VikasMeena-1/GEE/blob/main/IMD_Rain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================
# IMD Rainfall (2021–2025)
# District-wise Monthly Min, Max, Mean
# राजस्थान (DAUSA, KOTPUTLI-BEHROR, TONK)
# ============================================

# 1. Install libraries
!pip install geopandas xarray rioxarray netCDF4 pandas shapely

# 2. Import libraries
import geopandas as gpd
import xarray as xr
import rioxarray
import pandas as pd

# 3. Load shapefile
shp_path = "Rajasthan_districts.shp"

gdf = gpd.read_file(shp_path)
gdf = gdf.to_crs("EPSG:4326")

# Keep only your 3 districts (optional but safer)
gdf = gdf[gdf['DISTRICT'].isin(['DAUSA', 'KOTPUTLI-BEHROR', 'TONK'])]

print("Shapefile loaded:")
print(gdf[['DISTRICT']])

# 4. NetCDF files list
nc_files = [
    "RF25_ind2021_rfp25.nc",
    "RF25_ind2022_rfp25.nc",
    "RF25_ind2023_rfp25.nc",
    "RF25_ind2024_rfp25.nc",
    "RF25_ind2025_rfp25.nc"
]

# 5. Load and merge all NetCDF files
ds_list = []

for file in nc_files:
    ds = xr.open_dataset(file)
    # Rename 'TIME' coordinate to 'time' for consistency and correct concatenation
    if 'TIME' in ds.coords and 'TIME' in ds.dims: # Check if TIME exists as a coordinate and dimension
        ds = ds.rename({'TIME': 'time'})
    elif 'TIME' in ds.data_vars and 'TIME' in ds.dims: # If TIME is a data variable but acts as a time dim
        ds = ds.set_coords('TIME').rename({'TIME': 'time'})
    ds_list.append(ds)

# Concatenate along the 'time' dimension to create a single continuous time series
ds_all = xr.concat(ds_list, dim="time")

print("\nDataset structure:")
print(ds_all)

# 6. Select rainfall variable (IMD uses 'rf')
rain = ds_all['RAINFALL'] # Corrected 'rf' to 'RAINFALL'

# 7. Fix coordinates if needed
if 'longitude' in rain.dims:
    rain = rain.rename({'longitude': 'lon'})
if 'latitude' in rain.dims:
    rain = rain.rename({'latitude': 'lat'})

# 8. Convert time and filter
# The 'time' coordinate is now correctly set during concatenation (step 5).
# We just need to ensure its type is datetime and then select the range.
rain['time'] = pd.to_datetime(rain['time'].values) # Ensure 'time' coordinate is datetime objects
rain = rain.sel(time=slice('2021-01-01', '2025-12-31'))

# 9. Convert daily → monthly rainfall (mm)
rain_monthly = rain.resample(time='1M').sum()

# 10. Assign CRS
rain_monthly = rain_monthly.rio.write_crs("EPSG:4326")

# 11. District-wise zonal statistics
results = []

for idx, row in gdf.iterrows():
    district_name = row['DISTRICT']
    geom = [row.geometry]

    clipped = rain_monthly.rio.clip(geom, gdf.crs, drop=True)

    df = clipped.to_dataframe(name='rainfall').reset_index()

    # Remove no-data values
    df = df.dropna(subset=['rainfall'])

    stats = df.groupby('time')['rainfall'].agg(
        min='min',
        max='max',
        mean='mean'
    ).reset_index()

    stats['district'] = district_name

    results.append(stats)

# 12. Merge all districts
final_df = pd.concat(results, ignore_index=True)

# 13. Format output
final_df['year'] = final_df['time'].dt.year
final_df['month'] = final_df['time'].dt.month

final_df = final_df[['district', 'year', 'month', 'min', 'max', 'mean']]

# Sort for clean output
final_df = final_df.sort_values(['district', 'year', 'month'])

print("\nFinal Output Preview:")
print(final_df.head())

# 14. Export CSV
output_file = "Rajasthan_IMD_Monthly_Rainfall_2021_2025.csv"
final_df.to_csv(output_file, index=False)

from google.colab import files
files.download(output_file)

print("\n✅ CSV exported successfully!")

Shapefile loaded:
          DISTRICT
0            DAUSA
1  KOTPUTLI-BEHROR
2             TONK

Dataset structure:
<xarray.Dataset> Size: 127MB
Dimensions:    (time: 1826, LATITUDE: 129, LONGITUDE: 135)
Coordinates:
  * time       (time) datetime64[ns] 15kB 2021-01-01 2021-01-02 ... 2025-12-31
  * LATITUDE   (LATITUDE) float64 1kB 6.5 6.75 7.0 7.25 ... 38.0 38.25 38.5
  * LONGITUDE  (LONGITUDE) float64 1kB 66.5 66.75 67.0 ... 99.5 99.75 100.0
Data variables:
    RAINFALL   (time, LATITUDE, LONGITUDE) float32 127MB nan nan nan ... nan nan
Attributes:
    history:      FERRET V6.82   20-Feb-26
    Conventions:  CF-1.0


/usr/local/lib/python3.12/dist-packages/xarray/groupers.py:543: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(



Final Output Preview:
  district  year  month        min         max       mean
0    DAUSA  2021      1   7.375094   29.860306  18.959904
1    DAUSA  2021      2   0.000000    1.015082   0.169180
2    DAUSA  2021      3   6.709047   12.270744   8.701275
3    DAUSA  2021      4   0.000000    0.000000   0.000000
4    DAUSA  2021      5  54.226543  131.458344  79.271309


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ CSV exported successfully!
